# REGPLEX_Local — Reproducible Scientific Tutorial

This notebook reproduces the implemented REGPLEX v13 workflow for **Low Perplexity Region (LPR)** detection from FASTA input.


## 1. Introduction
REGPLEX identifies local sequence-complexity depressions in DNA using a deterministic, training-free pipeline.


In [ ]:
from regplex_core import *
from motif_engine import compile_motifs, annotate_regions
from visualization import (
    plot_perplexity_landscape,
    plot_smoothed_perplexity,
    plot_pds_landscape,
    plot_region_ranking,
    plot_three_window,
)
import pandas as pd


## 2. Theory
Dinucleotide perplexity measures local uncertainty in adjacent-base transitions. Sustained drops in perplexity relative to nearby context define candidate low-perplexity regions.


In [ ]:
workflow = [
    "DNA",
    "Dinucleotide Perplexity (17 nt)",
    "Savitzky–Golay Smoothing",
    "Perplexity Depression Score (PDS)",
    "Bounded Kadane Optimization",
    "Region Expansion",
    "Region Merging",
    "Low Perplexity Region Ranking",
    "Optional Motif Annotation",
    "Downloads",
]
workflow


## 3. Mathematical formulation
For upstream mean $\bar U$, region mean $\bar R$, and downstream mean $\bar D$:

$$
\mathrm{PDS}=\frac{\bar U + \bar D}{2}-\bar R
$$

Ranking score:

$$
\mathrm{RegionScore}=\mathrm{PDSMean}\times\mathrm{Persistence}\times\log(\mathrm{Length})\times\frac{1}{\mathrm{Variance}+\varepsilon}
$$


In [ ]:
print('PDS and RegionScore are computed directly in regplex_core.py')


## 4. Loading FASTA
Load input sequences and inspect record counts.


In [ ]:
with open('examples/ecoli.fasta', encoding='utf-8') as fh:
    fasta_text = fh.read()

records = parse_fasta(fasta_text)
len(records), records[0][0], len(records[0][1])


## 5. Dinucleotide perplexity
Compute the primary signal (window = 17 nt).


In [ ]:
sequence_id, seq = records[0]
di = compute_di_perplexity(seq, window=17)
len(di), float(pd.Series(di).dropna().mean())


## 6. Savitzky–Golay smoothing
Apply one smoothing pass to the perplexity profile.


In [ ]:
smoothed = smooth_perplexity(di, window_length=21, poly_order=3)
plot_smoothed_perplexity(di, smoothed, [])


## 7. PDS calculation
Compute local bilateral contrast against genomic flanks.


In [ ]:
pds = compute_pds(smoothed, flank_size=100, spacer_size=50, min_candidate=50, max_candidate=1000)
plot_pds_landscape(pds, [])


## 8. Region detection
Identify positive-PDS core regions using bounded Kadane optimization.


In [ ]:
cores = _find_all_pds_regions(pds, min_region_length=100, max_region_length=1000)
cores[:5], len(cores)


## 9. Expansion
Expand each core while local support remains above expansion rules.


In [ ]:
expanded = [_expand_region_pds(pds, s, e) for s, e in cores]
expanded[:5]


## 10. Merging
Merge nearby expanded regions separated by short gaps.


In [ ]:
merged = _merge_regions(expanded, gap=100)
merged[:5], len(merged)


## 11. Ranking
Compute region metrics and rank by RegionScore.


In [ ]:
result = analyze_sequence(sequence_id, seq)
df = pd.DataFrame(result.regions)
df[['Region_ID', 'Start', 'End', 'Length', 'PDSMean', 'RegionScore', 'Rank']].head()


## 12. Motif annotation
Annotate detected regions with optional motif patterns.


In [ ]:
motifs = compile_motifs('TATAWAWR\nCGCG')
annotated = annotate_regions(result.regions, motifs)
pd.DataFrame(annotated)[['Region_ID', 'MotifCount', 'Motifs']].head()


## 13. Export
Export tabular and interval outputs.


In [ ]:
export_df = pd.DataFrame(result.regions)
csv_bytes = export_table(export_df, 'csv')
bed_bytes = export_bed(export_df)
len(csv_bytes), len(bed_bytes)


## 14. Interpretation
Detected regions are algorithmic candidates defined by local complexity depression and ranking statistics. Biological significance requires independent experimental or comparative validation.


In [ ]:
plot_region_ranking(result.regions)


---
# 15. Population-Level Low Perplexity Region Analysis

When all sequences in a FASTA share **identical length**, REGPLEX automatically
activates **Population Analysis Mode**.  
This section demonstrates:

1. Loading a multi-sequence FASTA (aligned promoters / orthologous sequences)
2. Detecting LPRs in every sequence
3. Computing mean perplexity and mean PDS profiles
4. Identifying consensus LPRs (default: ≥50% support)
5. Building sequence × consensus-region occurrence matrix
6. Generating publication-quality figures (heatmap, LPR frequency, motif barplot)


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook'

from regplex_core import parse_fasta, analyze_sequence
from motif_engine import compile_motifs, annotate_regions
from population_analysis import (
    is_population_mode,
    compute_population_stats,
    compute_consensus_lprs,
    build_occurrence_matrix,
    population_summary_table,
    compute_motif_frequencies,
)
from visualization import (
    plot_mean_perplexity_profile,
    plot_mean_pds_profile,
    plot_lpr_frequency,
    plot_positional_heatmap,
    plot_consensus_lpr_track,
    plot_region_length_distribution,
    plot_region_score_distribution,
    plot_motif_frequency_barplot,
    plot_boundary_density,
)


### 15.1 Load multiple promoter sequences


In [ ]:
# Load the bundled population example (10 synthetic promoters, 2000 bp each)
with open('examples/population_example.fasta') as fh:
    fasta_text = fh.read()

records = parse_fasta(fasta_text)
print(f'Loaded {len(records)} sequences')
print(f'Lengths: {sorted(set(len(s) for _, s in records))}')


### 15.2 Detect LPRs in every sequence


In [ ]:
results = [analyze_sequence(header, seq) for header, seq in records]
print(f'Population mode active: {is_population_mode(results)}')
for r in results:
    print(f'  {r.sequence_id}: {len(r.regions)} LPR(s)')


### 15.3 Compute population statistics


In [ ]:
stats = compute_population_stats(results)
print(f'Signal length : {stats.signal_length}')
print(f'Sequences     : {stats.n_seq}')
print(f'Peak LPR freq : {stats.lpr_frequency.max():.2%}')
print(f'Total LPRs    : {len(stats.region_lengths)}')


### 15.4 Mean Perplexity Profile


In [ ]:
fig = plot_mean_perplexity_profile(stats)
fig.show()


### 15.5 Mean PDS Profile


In [ ]:
fig = plot_mean_pds_profile(stats)
fig.show()


### 15.6 Compute Consensus LPRs


In [ ]:
MIN_SUPPORT = 0.5  # 50% of sequences must contain an LPR at the position

consensus = compute_consensus_lprs(stats, results=results, min_support=MIN_SUPPORT)
print(f'Consensus LPRs at {MIN_SUPPORT:.0%} support: {len(consensus)}')

summary = population_summary_table(consensus)
summary


### 15.7 LPR Frequency Plot


In [ ]:
fig = plot_lpr_frequency(stats, consensus)
fig.show()


### 15.8 Positional Heatmap (Sequence × Position)


In [ ]:
# Perplexity heatmap
fig = plot_positional_heatmap(results, signal='perplexity')
fig.show()


### 15.9 Consensus LPR Track


In [ ]:
fig = plot_consensus_lpr_track(stats, consensus)
fig.show()


### 15.10 Region Occurrence Matrix


In [ ]:
occ_matrix = build_occurrence_matrix(results, consensus)
print('Occurrence matrix (rows=sequences, cols=consensus LPRs):')
occ_matrix


### 15.11 Region Length and Score Distributions


In [ ]:
fig = plot_region_length_distribution(stats)
fig.show()

fig = plot_region_score_distribution(stats)
fig.show()


### 15.12 Region Boundary Density


In [ ]:
fig = plot_boundary_density(stats)
fig.show()


### 15.13 Motif Frequency Enrichment


In [ ]:
MOTIFS = 'TATAWAWR\nA{7}|T{7}\nT{7}\nATATAT\n'
compiled = compile_motifs(MOTIFS)

# Annotate all individual sequence regions
for r in results:
    annotate_regions(r.regions, compiled)

# Compute per-consensus-region motif frequency
consensus = compute_motif_frequencies(results, consensus, compiled)

for c in consensus:
    print(f'{c.region_id}: {c.motif_frequencies}')

fig = plot_motif_frequency_barplot(consensus)
fig.show()


### 15.14 Export Population Results

All outputs can be downloaded as CSV/Excel.


In [ ]:
# Save population summary
summary.to_csv('regplex_population_summary.csv', index=False)
print('Saved: regplex_population_summary.csv')

# Save occurrence matrix
occ_matrix.to_csv('regplex_occurrence_matrix.csv')
print('Saved: regplex_occurrence_matrix.csv')

# Save all individual region results
from regplex_core import regions_dataframe
all_df = regions_dataframe(results)
all_df.to_csv('regplex_all_regions.csv', index=False)
print(f'Saved {len(all_df)} total regions to: regplex_all_regions.csv')
